# SITBIO · Análisis inicial de datos — Semana 2 (Ciencia de Datos con Python)

**Aprendiz**: Esteban Bedoya Rojo · **Programa**: ADSO — Centro CTMA

Objetivo: aplicar las **estructuras básicas de Python, NumPy y Pandas** sobre
**datos reales del proyecto SITBIO** (equipos biomédicos e intervenciones exportados
de la base PostgreSQL del modo Api), inspeccionarlos, procesarlos y exportarlos.

| Sección | Contenido |
|---|---|
| 1 | Entorno y librerías |
| 2 | Listas y diccionarios (con reto) |
| 3 | NumPy |
| 4 | Series y DataFrames |
| 5 | Lectura de CSV y Excel (rutas relativas) |
| 6 | Inspección inicial |
| 7 | Hallazgos iniciales |
| 8 | Proceso: columnas calculadas |
| 9 | Exportación |
| 10 | Conclusiones |

## 1. Entorno y librerías

Todas las rutas son **relativas** a la carpeta del notebook (`src/analytics/notebooks`).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Rutas SIEMPRE relativas (buena práctica exigida por la guía; nada de rutas absolutas)
BASE = Path("..")            # src/analytics
RAW = BASE / "data" / "raw"
PROCESADO = BASE / "data" / "processed"

print(f"pandas {pd.__version__} · numpy {np.__version__}")
print("Ruta de datos crudos:", RAW)

pandas 3.0.6 · numpy 2.5.3
Ruta de datos crudos: ..\data\raw


## 2. Listas y diccionarios

Lista de nombres, diccionario de un registro y lista de diccionarios, con datos de SITBIO.

In [2]:
# Lista de nombres (técnicos de la siembra de SITBIO)
tecnicos = ["Laura Gómez Ríos", "Carlos Méndez Ruiz", "Ana Patricia Castro", "Jorge Luis Ramírez"]

# Diccionario: UN registro (equipo biomédico)
equipo = {"codigo": "BM-00001", "nombre": "Monitor multiparamétrico", "marca": "Getinge", "estado": 0}

# Lista de diccionarios: varios equipos
equipos_demo = [
    {"codigo": "BM-00001", "nombre": "Monitor multiparamétrico", "marca": "Getinge"},
    {"codigo": "BM-00002", "nombre": "Bomba de infusión", "marca": "Mindray"},
]

# Reto: añadir 2 registros a la lista
equipos_demo.append({"codigo": "BM-00003", "nombre": "Tomógrafo 128 cortes", "marca": "Siemens"})
equipos_demo.append({"codigo": "BM-00004", "nombre": "Autoclave horizontal 340 L", "marca": "Steris"})

print(f"Técnicos: {len(tecnicos)} · Equipos demo: {len(equipos_demo)}")
for e in equipos_demo:
    print(f"  {e['codigo']} — {e['nombre']} ({e['marca']})")

Técnicos: 4 · Equipos demo: 4
  BM-00001 — Monitor multiparamétrico (Getinge)
  BM-00002 — Bomba de infusión (Mindray)
  BM-00003 — Tomógrafo 128 cortes (Siemens)
  BM-00004 — Autoclave horizontal 340 L (Steris)


## 3. NumPy

Arreglos con **valores reales** de `data/raw/intervenciones.xlsx` (tenant 1 de SITBIO): horas de intervención y costo de mano de obra.

In [3]:
# Valores reales tomados de la exportación de intervenciones de SITBIO
horas = np.array([8.2, 5.2, 5.5, 5.5, 2.1, 7.3, 5.0, 1.8, 2.9])
mano_obra = np.array([938682.0, 487534.0, 389132.0, 326820.0, 746950.0, 1045582.0, 771487.0, 301880.0, 871949.0])

print("promedio horas:", round(horas.mean(), 2))
print("máximo:", horas.max(), "· mínimo:", horas.min())
print("total:", horas.sum(), "· desviación estándar:", round(horas.std(), 2))

# Filtro vectorizado: intervenciones con más de 5 horas
print("horas > 5:", horas[horas > 5])

# Operaciones vectorizadas: SLA de referencia 4 h y factor 1.5 sobre la mano de obra
sla = np.where(horas <= 4, "dentro del SLA", "fuera del SLA")
costo_estimado = mano_obra * 1.5
print("SLA:", sla.tolist())
print("costo estimado (x1.5), primeros 3:", costo_estimado[:3])

promedio horas: 4.83
máximo: 8.2 · mínimo: 1.8
total: 43.49999999999999 · desviación estándar: 2.08
horas > 5: [8.2 5.2 5.5 5.5 7.3]
SLA: ['fuera del SLA', 'fuera del SLA', 'fuera del SLA', 'fuera del SLA', 'dentro del SLA', 'fuera del SLA', 'fuera del SLA', 'dentro del SLA', 'dentro del SLA']
costo estimado (x1.5), primeros 3: [1408023.  731301.  583698.]


## 4. Series y DataFrames

Creación desde diccionarios y listas, e identificación de filas, columnas e índices.

In [4]:
# Serie: horas por técnico (índice = nombre del técnico)
horas_por_tecnico = pd.Series(
    {"Camilo Restrepo Ruiz": 4.1, "Andrés Montoya Vélez": 4.7, "Sin asignar": 5.5},
    name="horas_promedio",
)
print(horas_por_tecnico)
print("índice:", horas_por_tecnico.index.tolist())

# DataFrame desde una lista de diccionarios (órdenes reales)
df_demo = pd.DataFrame([
    {"orden": "SIT-2026-000001", "horas": 8.2, "tecnico": "Sin asignar"},
    {"orden": "SIT-2026-000005", "horas": 5.2, "tecnico": "Camilo Restrepo Ruiz"},
    {"orden": "SIT-2026-000014", "horas": 2.1, "tecnico": "Andrés Montoya Vélez"},
])
print(df_demo)
print("filas, columnas (shape):", df_demo.shape)
print("columnas:", df_demo.columns.tolist())
print("índice:", df_demo.index.tolist())

Camilo Restrepo Ruiz    4.1
Andrés Montoya Vélez    4.7
Sin asignar             5.5
Name: horas_promedio, dtype: float64
índice: ['Camilo Restrepo Ruiz', 'Andrés Montoya Vélez', 'Sin asignar']
             orden  horas               tecnico
0  SIT-2026-000001    8.2           Sin asignar
1  SIT-2026-000005    5.2  Camilo Restrepo Ruiz
2  SIT-2026-000014    2.1  Andrés Montoya Vélez
filas, columnas (shape): (3, 3)
columnas: ['orden', 'horas', 'tecnico']
índice: [0, 1, 2]


## 5. Lectura de CSV y Excel

`pd.read_csv` y `pd.read_excel` con **rutas relativas** (`Path("..")`) sobre los archivos de `data/raw`.

In [5]:
equipos_df = pd.read_csv(RAW / "equipos.csv")                    # exportado de PostgreSQL
intervenciones_df = pd.read_excel(RAW / "intervenciones.xlsx")   # exportado de PostgreSQL

print("equipos.csv →", equipos_df.shape, "filas × columnas")
print("intervenciones.xlsx →", intervenciones_df.shape, "filas × columnas")
equipos_df.head(3)

equipos.csv → (35, 11) filas × columnas
intervenciones.xlsx → (9, 11) filas × columnas


,CodigoInventario,Nombre,Categoria,Marca,Modelo,Ubicacion,Responsable,Estado,FechaInstalacion,ValorAdquisicion,VidaUtilMeses
0,BM-00001,Monitor multiparamétrico,Monitor multiparamétrico,Getinge,MX-536B,Laboratorio central,Ingeniería biomédica,0,2024-04-24 00:00:00,176594950.0,93
1,BM-00002,Bomba de infusión,Bomba de infusión,Mindray,MX-398E,Neonatología,Servicio externo,0,2019-04-18 00:00:00,115865309.0,110
2,BM-00003,Desfibrilador,Desfibrilador,Siemens Healthineers,MX-465F,Laboratorio central,Ingeniería biomédica,0,2020-08-12 00:00:00,40809385.0,107


## 6. Inspección inicial

`head`, `tail`, `shape`, `columns`, `dtypes`, `info()`, `describe()` y `describe(include="all")`.

In [6]:
print("shape:", equipos_df.shape)
print("columns:", equipos_df.columns.tolist())
print("dtypes:")
print(equipos_df.dtypes)
print()
equipos_df.info()

shape: (35, 11)
columns: ['CodigoInventario', 'Nombre', 'Categoria', 'Marca', 'Modelo', 'Ubicacion', 'Responsable', 'Estado', 'FechaInstalacion', 'ValorAdquisicion', 'VidaUtilMeses']
dtypes:
CodigoInventario        str
Nombre                  str
Categoria               str
Marca                   str
Modelo                  str
Ubicacion               str
Responsable             str
Estado                int64
FechaInstalacion        str
ValorAdquisicion    float64
VidaUtilMeses         int64
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CodigoInventario  35 non-null     str    
 1   Nombre            35 non-null     str    
 2   Categoria         35 non-null     str    
 3   Marca             35 non-null     str    
 4   Modelo            35 non-null     str    
 5   Ubicacion         35 non-null     str    
 6   Responsable  

In [7]:
print("--- describe(): estadísticas numéricas ---")
print(equipos_df.describe())
print()
print('--- describe(include="all"): también las columnas de texto ---')
print(equipos_df.describe(include="all"))

--- describe(): estadísticas numéricas ---
          Estado  ValorAdquisicion  VidaUtilMeses
count  35.000000      3.500000e+01      35.000000
mean    0.600000      1.017087e+08      89.885714
std     1.193118      5.025512e+07      16.997874
min     0.000000      1.528382e+07      60.000000
25%     0.000000      5.168129e+07      75.500000
50%     0.000000      1.096832e+08      93.000000
75%     0.500000      1.345341e+08     100.500000
max     4.000000      1.804394e+08     120.000000

--- describe(include="all"): también las columnas de texto ---
       CodigoInventario                    Nombre                 Categoria  \
count                35                        35                        35   
unique               35                        17                        16   
top            BM-00001  Monitor multiparamétrico  Monitor multiparamétrico   
freq                  1                         3                         4   
mean                NaN                       Na

In [8]:
print("--- head (primeras 3) y tail (últimas 3) ---")
print(equipos_df.head(3))
print(equipos_df.tail(3))

--- head (primeras 3) y tail (últimas 3) ---
  CodigoInventario                    Nombre                 Categoria  \
0         BM-00001  Monitor multiparamétrico  Monitor multiparamétrico   
1         BM-00002         Bomba de infusión         Bomba de infusión   
2         BM-00003             Desfibrilador             Desfibrilador   

                  Marca   Modelo            Ubicacion           Responsable  \
0               Getinge  MX-536B  Laboratorio central  Ingeniería biomédica   
1               Mindray  MX-398E         Neonatología      Servicio externo   
2  Siemens Healthineers  MX-465F  Laboratorio central  Ingeniería biomédica   

   Estado     FechaInstalacion  ValorAdquisicion  VidaUtilMeses  
0       0  2024-04-24 00:00:00       176594950.0             93  
1       0  2019-04-18 00:00:00       115865309.0            110  
2       0  2020-08-12 00:00:00        40809385.0            107  
   CodigoInventario                                     Nombre  \
32         

## 7. Hallazgos iniciales

- **Filas y columnas**: `equipos.csv` tiene **35 filas × 11 columnas**; `intervenciones.xlsx` tiene **9 × 11**.
- **Tipos observados**: códigos y textos (`object`), fechas guardadas como **texto** (`object`),
  `ValorAdquisicion` y costos como decimales, `Estado` como **entero** (enum del cliente:
  0 Operativo · 1 En mantenimiento · 2 En calibración · 3 Fuera de servicio · 4 Baja).
- **Columna a revisar**: `Estado` está **sin etiqueta** (es un código numérico) — hay que mapearla
  a texto antes de cualquier análisis; igual `FechaInstalacion`, que debe convertirse a fecha.
- **Ojo con la moneda**: `ValorAdquisicion` está en pesos colombianos (valores del orden de
  $66 millones a $190 millones) — no debe tratarse como un conteo.
- **Pregunta analítica**: ¿las intervenciones de **más de 5 horas** concentran mayor costo de mano
  de obra, y qué técnico promedia más horas por intervención?

## 8. Proceso: columnas calculadas

Se prepara el dataset procesado con columnas nuevas (equivalente a `promedio_general`/`estado_academico` del ejemplo de la guía).

In [9]:
# Columnas calculadas — equipos
ESTADOS = {0: "Operativo", 1: "En mantenimiento", 2: "En calibración", 3: "Fuera de servicio", 4: "Baja"}

equipos_pro = equipos_df.copy()
equipos_pro["estado_equipo"] = equipos_pro["Estado"].map(ESTADOS)          # enum → texto
equipos_pro["FechaInstalacion"] = pd.to_datetime(equipos_pro["FechaInstalacion"])
equipos_pro["antiguedad_anios"] = ((pd.Timestamp.today() - equipos_pro["FechaInstalacion"]).dt.days / 365.25).round(1)

# Columnas calculadas — intervenciones
inter_pro = intervenciones_df.copy()
inter_pro["duracion_cumple_sla"] = np.where(inter_pro["HorasInvertidas"] <= 4, "Sí", "No")
promedio_por_tecnico = inter_pro.groupby("TecnicoNombre")["HorasInvertidas"].mean().round(2)
inter_pro["horas_promedio_tecnico"] = inter_pro["TecnicoNombre"].map(promedio_por_tecnico)   # columnas calculadas: horas_promedio_tecnico

print(equipos_pro[["CodigoInventario", "estado_equipo", "antiguedad_anios"]].head(4))
print()
print("horas promedio por técnico:")
print(promedio_por_tecnico)
inter_pro[["OrdenCodigo", "TecnicoNombre", "HorasInvertidas", "duracion_cumple_sla", "horas_promedio_tecnico"]].head(4)

  CodigoInventario estado_equipo  antiguedad_anios
0         BM-00001     Operativo               2.5
1         BM-00002     Operativo               7.5
2         BM-00003     Operativo               6.2
3         BM-00004     Operativo               4.9

horas promedio por técnico:
TecnicoNombre
Andrés Montoya Vélez    4.70
Camilo Restrepo Ruiz    4.38
Sin asignar             5.53
Name: HorasInvertidas, dtype: float64


,OrdenCodigo,TecnicoNombre,HorasInvertidas,duracion_cumple_sla,horas_promedio_tecnico
0,SIT-2026-000001,Sin asignar,8.2,No,5.53
1,SIT-2026-000005,Camilo Restrepo Ruiz,5.2,No,4.38
2,SIT-2026-000002,Sin asignar,5.5,No,5.53
3,SIT-2026-000004,Camilo Restrepo Ruiz,5.5,No,4.38


## 9. Exportación a `data/processed`

El dataset procesado se guarda en **CSV y Excel** (columnas calculadas incluidas).

In [10]:
salida_csv = PROCESADO / "equipos_procesado.csv"
salida_xlsx = PROCESADO / "intervenciones_procesado.xlsx"

equipos_pro.to_csv(salida_csv, index=False, encoding="utf-8")
inter_pro.to_excel(salida_xlsx, index=False)
print("Exportado:", salida_csv, "→", salida_csv.stat().st_size, "bytes")
print("Exportado:", salida_xlsx, "→", salida_xlsx.stat().st_size, "bytes")

# Verificación: relectura del CSV exportado
verificacion = pd.read_csv(PROCESADO / "equipos_procesado.csv")
print("relectura CSV:", verificacion.shape, "· ¿tiene estado_equipo?", "estado_equipo" in verificacion.columns)

Exportado: ..\data\processed\equipos_procesado.csv → 5327 bytes
Exportado: ..\data\processed\intervenciones_procesado.xlsx → 5979 bytes
relectura CSV: (35, 13) · ¿tiene estado_equipo? True


## 10. Conclusiones

**Estructuras usadas y para qué**

- **Lista** (`tecnicos`): colección ordenada de elementos iguales — se recorre con `for` y se mide con `len()`.
- **Diccionario** (`equipo`): pares clave–valor para **un** registro con campos con nombre.
- **Lista de diccionarios** (`equipos_demo`): la forma natural de datos tabulares antes de Pandas.
- **ndarray de NumPy** (`horas`, `mano_obra`): operaciones matemáticas y filtros **vectorizados** sin bucles.
- **Serie** (`horas_por_tecnico`): una columna con **índice etiquetado**.
- **DataFrame** (`equipos_df`): la tabla completa: filas (registros), columnas (variables) e índice.

**Diferencias clave lista / diccionario / Serie / DataFrame**

| Estructura | Dimensión | Acceso | Ejemplo aquí |
|---|---|---|---|
| Lista | 1D sin etiquetas | por posición | nombres de técnicos |
| Diccionario | clave–valor | por clave | un equipo |
| Serie | 1D con índice | por etiqueta o posición | horas promedio por técnico |
| DataFrame | 2D con índice y columnas | `df[col]`, `.loc`, `.iloc` | equipos e intervenciones |

**Qué entrega cada comando de inspección**: `shape` → dimensiones; `columns` → nombres de
variables; `dtypes` → tipo de cada columna; `info()` → tipos + no nulos (memoria); `describe()`
→ estadísticas de columnas numéricas; `describe(include="all")` → suma también las categóricas
(frecuencia del valor más repetido y n únicos).

**Dos buenas prácticas aplicadas**

1. **Rutas relativas con `pathlib`** (`Path("..") / "data" / "raw"`): el notebook funciona en
   cualquier equipo sin editar rutas absolutas.
2. **Separar datos crudos de procesados**: `data/raw` guarda los originales exportados de SITBIO y
   `data/processed` solo lo generado por el notebook, que siempre se puede regenerar ejecutando
   todo el cuaderno de principio a fin.

*Generado para la evidencia de la Semana 2 del programa ADSO (Centro CTMA).*